In [1]:
from thermopy.eos import PengRobinson, VanDerWaals
from thermopy.properties import fugacity
from scipy.optimize import brentq
import numpy as np

# Finding $P_{sat}$ from fugacity coefficients

The saturated pressure of a fluid at a temperature T can be found through the criteria: $\phi^v = \phi^l$
Through varying the pressure of a system we can identify when this criteria is met giving the saturation pressure

Methanes saturation temperature at 101325 Pa (1 atm) is known to be 111.66K, we can attempt to find this saturation pressure using a cubic equation of state to verify its accuracy

In [2]:
"""
Finds the saturated pressure at a given temperature using an equation of state
uses a Clausius Clapeyron relation to generate an estimate for bound setting
"""


#Instantiate our EoS object
methane = PengRobinson("methane")

Temperature = 111.66 #[K]
C = 5.5 #Clasius-Clapeyron constant for simple fluids
tr = Temperature/methane.Tc
p_rest = C*(1-1/tr)
p_est = np.exp(p_rest) * methane.Pc #estimated Psat to generate bounds for solver

#Define objective function
def objectivef(p):
    state = methane.solve(T=Temperature, P=p)

    #Assign each phase to its results
    liquid = state[0]
    vapour = state[-1]

    liq_phi = fugacity.coefficient(liquid, methane, cubicEoS=True)
    vap_phi = fugacity.coefficient(vapour, methane, cubicEoS=True)
    #function to find root of
    return liq_phi-vap_phi


Psat = brentq(objectivef, 0.5*p_est, 3*p_est)

print(f"Saturation pressure of methane at {Temperature}K is {Psat}")

Saturation pressure of methane at 111.66K is 102014.87833011513
